In [ ]:
# 필요한 도구(라이브러리)
import requests # 웹페이지에 접속해서 데이터를 가져오는 도구
from bs4 import BeautifulSoup # 가져온 HTML 문서를 분석하기 쉽게 쪼개주는 도구
from io import BytesIO # 컴퓨터 메모리 상에 가상의 파일을 만들어주는 도구
from PIL import Image # 파이썬에서 이미지를 다루고 저장하는 도구
import re # 문자열에서 특정 패턴을 찾거나 바꾸는 정규표현식 도구
import os # 컴퓨터의 폴더나 파일을 다루는 운영체제 도구
from urllib.parse import urlparse, parse_qs # 인터넷 주소(URL)를 구조별로 쪼개고 분석하는 도구

# 영화 랭킹 정보 조회 및 포스터 이미지 저장하기
- 상대 경로 방식 : 가져온 src(소스) 문자열 앞에 "https://www.moviechart.co.kr" 도메인을 문자열 더하기(+)로 단순 결합하여 다운로드 링크를 만듦

In [ ]:
# 1. 무비차트 실시간 랭킹 페이지의 전체 데이터(HTML)를 서버에 요청해서 가져옵니다.
movie_ranking = requests.get("https://www.moviechart.co.kr/rank/realtime/index/image")

# 2. 이미지를 저장할 폴더 이름을 'images'로 지정합니다.
image_dir = 'images'

# 3. 만약 현재 위치에 'images'라는 이름의 폴더가 존재하지 않는다면(False)
if not os.path.exists(image_dir):
  # 'images' 폴더를 새로 만듭니다.
  os.makedirs(image_dir)

# 4. 윈도우 등의 운영체제에서 파일 이름에 쓸 수 없는 특수문자들을 찾기 위한 규칙을 만듭니다.
pattern = r'[\\/:"*?<>|]' #파일이나 폴더 이름으로 절대 사용할 수 없는 특수문자 9가지

# 5. 서버에 요청한 결과가 '200(성공)'이라면 아래 코드를 실행합니다.
if movie_ranking.status_code == 200:
  print("영화 정보를 출력합니다.")
  
  # 6. 가져온 텍스트(HTML) 데이터를 파이썬이 다루기 쉽도록 BeautifulSoup 객체로 변환합니다.
  soup = BeautifulSoup(movie_ranking.content, 'html.parser')
  
  # 주석: (개발자 도구를 통해 복사한 HTML 요소의 경로 참고용)
  #content > div.wArea.space > div.movieBox > ul > li:nth-child(2) > div > div.movie-title > h3 > a
  
  # 7. HTML 안에서 조건(클래스명 등)에 맞는 영화 제목 요소(<a> 태그)들을 모두 찾아 리스트로 만듭니다.
  movie_title_list = soup.select(".movieBox-list .movie-title a") # 영화 이름 <a> 요소 목록
  
  # 8. HTML 안에서 조건에 맞는 영화 포스터 이미지 요소(<img> 태그)들을 모두 찾아 리스트로 만듭니다.
  movie_image_list = soup.select(".movieBox-list .movieBox-item img") #img 태그 요소
  
  # 9. 찾아낸 영화 제목의 개수를 세어서 출력합니다.
  print(f"수집한 영화 수: {len(movie_title_list)}")

  # 10. 제목 리스트와 이미지 리스트에서 하나씩 짝지어서 꺼내며 반복문을 실행합니다.
  for movie_title, movie_image in zip(movie_title_list, movie_image_list):
    # 11. 화면에 영화 제목 텍스트와 이미지의 URL 주소를 출력해 봅니다.
    print(movie_title.text, movie_image.get('src'))  #이름 요소의 텍스트, 이미지 요소의 src속성의 값
    
    # 12. 이미지 태그에서 주소(src)만 따로 변수에 저장합니다.
    image_src = movie_image.get('src')
    
    # 13. 기본 웹사이트 주소와 이미지 주소를 합쳐서 실제 이미지가 있는 곳으로 다시 요청을 보냅니다.
    image_response = requests.get("https://www.moviechart.co.kr" + image_src)
    
    # 14. 인터넷에서 다운받은 이미지 데이터(바이트)를 메모리상에서 열쇠(이미지 객체)로 변환해 엽니다.
    img = Image.open(BytesIO(image_response.content))
    
    # 15. 영화 제목에 파일 이름으로 쓸 수 없는 특수문자가 있다면 빈칸('')으로 바꿔서 지워버립니다.
    image_filename = re.sub(pattern, '', movie_title.text) #파일이름 지정, 정규식
    
    # 16. 완성된 파일 이름 뒤에 '.png'를 붙이고, 'images' 폴더 경로와 합쳐서 이미지를 저장합니다.
    img.save(os.path.join(image_dir, image_filename + ".png"))
    
    # 17. 저장이 완료된 영화 제목을 한 번 더 출력합니다.
    print(movie_title.text, )  
    
# 18. 만약 서버 요청 결과가 200(성공)이 아니라면 (예: 404 에러 등)
else:
  # 연결 실패 메시지를 출력합니다.
  print("페이지에 연결할 수 없습니다.")

영화 정보를 출력합니다.
수집한 영화 수: 20
왕과 사는 남자 /thumb?width=178&height=267&m_code=20242837&source=https://admin.moviechart.co.kr/assets/upload/movie/260108003526_8322.jpg
왕과 사는 남자
휴민트 /thumb?width=178&height=267&m_code=20241266&source=https://admin.moviechart.co.kr/assets/upload/movie/260112024651_6301.jpg
휴민트
초속 5센티미터 /thumb?width=178&height=267&m_code=20259583&source=https://admin.moviechart.co.kr/assets/upload/movie/260213052538_7190.jpg
초속 5센티미터
너자 2 /thumb?width=178&height=267&m_code=20261181&source=https://admin.moviechart.co.kr/assets/upload/movie/260202063756_4036.jpg
너자 2
슬라이드 스트럼 뮤트 /thumb?width=178&height=267&m_code=20261450&source=https://admin.moviechart.co.kr/assets/upload/movie/260219042530_2928.jpg
슬라이드 스트럼 뮤트
넘버원 /thumb?width=178&height=267&m_code=20252373&source=https://admin.moviechart.co.kr/assets/upload/movie/260123065552_5690.jpg
넘버원
햄넷 /thumb?width=178&height=267&m_code=20250365&source=https://admin.moviechart.co.kr/assets/upload/movie/260202063420_3704.jpg
햄넷
부흥 /thumb?wid

# 영화 포스터 수집 예에서 포스터 원본을 저장
- 쿼리 파라미터 파싱 방식:(작게 줄여진 썸네일 이미지x) source= 뒤에 적혀있는 실제 원본 고화질 이미지의 URL을 찾아 다운로드

In [ ]:


# 2. 무비차트 실시간 랭킹 페이지에 접속을 요청합니다.
movie_ranking = requests.get("https://www.moviechart.co.kr/rank/realtime/index/image")

# 3. 이번에는 이미지를 저장할 폴더 이름을 'images2'로 지정합니다.
image_dir = 'images2'

# 4. 'images2' 폴더가 없으면 새로 만듭니다.
if not os.path.exists(image_dir):
  os.makedirs(image_dir)

# 5. 파일 이름으로 쓸 수 없는 특수문자 9가지를 지정합니다.
pattern = r'[\\/:"*?<>|]' 

# 6. 웹페이지 접속에 성공(상태 코드 200)했다면 아래 작업을 시작합니다.
if movie_ranking.status_code == 200:
  print("영화 정보를 출력합니다.")
  
  # 7. 가져온 웹페이지 데이터를 HTML 구조로 분석할 준비를 합니다.
  soup = BeautifulSoup(movie_ranking.content, 'html.parser')
  
  # 8. 영화 제목이 담긴 <a> 태그와 포스터가 담긴 <img> 태그를 리스트로 모두 찾아냅니다.
  movie_title_list = soup.select(".movieBox-list .movie-title a")
  movie_image_list = soup.select(".movieBox-list .movieBox-item img")
  print(f"수집한 영화 수: {len(movie_title_list)}")

  # 9. 제목 리스트와 이미지 리스트를 하나씩 짝지어서 반복문을 돌립니다.
  for movie_title, movie_image in zip(movie_title_list, movie_image_list):
    
    # 10. 이미지 태그 안의 'src'(주소) 값을 문자열로 가져옵니다.
    url = movie_image.get('src')
    
    # 11. 가져온 주소(url)를 프로토콜, 도메인, 경로, 쿼리 등으로 쪼갭니다(파싱).
    parsed_url = urlparse(url)
    
    # 12. 쪼개진 부분 중 물음표(?) 뒤에 오는 '쿼리(query)' 부분만 딕셔너리 형태로 변환합니다.
    query_params = parse_qs(parsed_url.query)
    
    # 13. 변환된 쿼리에서 'source'라는 이름의 파라미터 값을 찾아 꺼냅니다. (원본 이미지 주소)
    # 만약 'source'가 없다면 None을 반환하도록 안전장치를 걸어둡니다.
    image_src = query_params.get('source', [None])[0]
    
    # 14. (주석 처리된 코드) 이전 코드처럼 도메인을 결합하지 않습니다.
    # image_response = requests.get('https://www.moviechart.co.kr' + image_src)
    
    # 15. 찾아낸 진짜 원본 이미지 주소(image_src)로 다시 접속을 요청해 이미지를 다운로드합니다.
    image_response = requests.get(image_src)
    
    # 16. 다운받은 이미지 데이터를 열어서 이미지 객체로 만듭니다.
    img = Image.open(BytesIO(image_response.content))
    
    # 17. 영화 제목에서 파일 이름에 쓸 수 없는 특수문자를 제거합니다.
    image_filename = re.sub(pattern, '', movie_title.text)
    
    # 18. 'images2' 폴더 안에 영화제목.png 이름으로 이미지를 최종 저장합니다.
    img.save(os.path.join(image_dir, image_filename + '.png'))
    
    # 19. 저장이 완료된 영화 제목을 화면에 출력합니다.
    print(movie_title.text, )
else:
  print("페이지에 연결할 수 없습니다.")